In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

import os, sys
os.chdir('..')

from sazz.samplers.AutomaticBoomerangSampler import AutomaticBoomerangSampler
from sazz.samplers.StickyAutomaticBoomerangSampler import StickyAutomaticBoomerangSampler
from sazz.models.bnn_torch import make_bnn_regression

## UCI

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

N_RESAMPLE = 50_000

# ── 1. Boston Housing ────────────────────────────────────────────
# 506 samples, 13 features, target = median home value (MEDV)
# sklearn removed load_boston; use the original CSV from StatLib
from sklearn.datasets import fetch_openml
boston_raw = fetch_openml(name="boston", version=1, as_frame=True, parser="auto")
X_boston = boston_raw.data.values.astype(float)
y_boston = boston_raw.target.values.astype(float)

# ── 2. Naval Propulsion Plants ───────────────────────────────────
# 11,934 samples, 16 features, target = GT compressor decay coeff (col 17)
df_naval = pd.read_csv("benchmarks_august/datasets/naval_data.txt",
                        sep=r"\s+", header=None)
X_naval = df_naval.iloc[:, :16].values.astype(float)
y_naval = df_naval.iloc[:, 16].values.astype(float)
n_naval = 1000 
rng = np.random.default_rng(42)
idx = rng.choice(len(df_naval), n_naval, replace=False)
X_naval, y_naval = X_naval[idx], y_naval[idx]

# ── 3. Energy Efficiency ─────────────────────────────────────────
# 768 samples, 8 features, target = Y1 (heating load)
df_energy = pd.read_excel("benchmarks_august/datasets/energy_data.xlsx")
X_energy = df_energy.iloc[:, :8].values.astype(float)
y_energy = df_energy.iloc[:, 8].values.astype(float)

# ── Standardize & split (Hernández-Lobato & Adams convention) ────
datasets = {}
for name, X, y in [("boston", X_boston, y_boston),
                    ("naval",  X_naval,  y_naval),
                    ("energy", X_energy, y_energy)
                    ]:
    # 90/10 train-test split
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    # Standardize using training statistics
    x_mean, x_std = X_tr.mean(axis=0), X_tr.std(axis=0)
    x_std[x_std == 0] = 1.0   # guard against constant columns (Naval has some)
    y_mean, y_std = y_tr.mean(), y_tr.std()

    X_tr = (X_tr - x_mean) / x_std
    X_te = (X_te - x_mean) / x_std
    y_tr = (y_tr - y_mean) / y_std
    y_te = (y_te - y_mean) / y_std

    datasets[name] = {
        "X_train": torch.tensor(X_tr, dtype=torch.float64),
        "y_train": torch.tensor(y_tr, dtype=torch.float64),
        "X_test":  torch.tensor(X_te, dtype=torch.float64),
        "y_test":  torch.tensor(y_te, dtype=torch.float64),
        # keep numpy versions for sklearn metrics if needed
        "x_mean": x_mean, "x_std": x_std,
        "y_mean": y_mean, "y_std": y_std,
    }
    print(f"{name:8s}  N={X.shape[0]:>6d}  D={X.shape[1]:>2d}  "
          f"train={X_tr.shape[0]}  test={X_te.shape[0]}")

## BOSTON

In [ ]:
from sazz.utils.bnn_utils import make_kappa_vector_bnn, make_kappa_from_inclusion
from sazz.utils.warmup import warmup, tune_refresh_rate
target_boston = make_bnn_regression(
    datasets['boston']['X_train'], datasets['boston']['y_train'],
    layer_sizes=[13, 50, 50, 1],
    activation="tanh",
    prior_std_weight=torch.sqrt(torch.tensor([2.0])),      # tight global weights
    prior_std_bias=1.0,        # loose biases
    fan_in_scaling=True,      # not scaling, since prior_std_weight=0.3 is already tight
    noise_std=0.5,             # appropriate noise assumption
    covariance_reference="laplace_diag",
    adam_steps=3000
)

In [ ]:
sampler_boston = AutomaticBoomerangSampler(
    grad_target=target_boston.grad_target,
    D=target_boston.D,
    thinning="pli",
    refresh_rate=0.1,
)
sampler_boston.preprocess(
    x_ref=target_boston.x_ref,
    Sigma_inv=target_boston.Sigma_inv
)

warmup(sampler_boston, n_rounds=5, n_pilot=200, target=target_boston)

tune_info = tune_refresh_rate(sampler_boston, n_pilot=200)
print(f"Tuned refresh rate: {tune_info['lambda_r_old']:.3f} -> "
      f"{tune_info['lambda_r_new']:.3f} (floor active: {tune_info['floor_active']})")


# Kappa per layer — middle layers sparser
kappa_boston = make_kappa_from_inclusion(layer_sizes=[13, 50, 50, 1], prior_std_weight=1.0,
                                  prior_inclusion_weight=0.5,  # sparsity belief
                                  fan_in_scaling=True)

sampler_boston_sticky = StickyAutomaticBoomerangSampler(
    grad_target=target_boston.grad_target,
    D=target_boston.D,
    thinning="pli",
    refresh_rate=1.0,
    kappa=kappa_boston
)

# Reference matches the prior — principled and needs no tuning
sampler_boston_sticky.preprocess(
    x_ref=target_boston.x_ref,
    Sigma_inv=target_boston.Sigma_inv
)

tune_info_sticky = tune_refresh_rate(sampler_boston_sticky, n_pilot=200)
print(f"Tuned refresh rate: {tune_info_sticky['lambda_r_old']:.3f} -> "
      f"{tune_info_sticky['lambda_r_new']:.3f} (floor active: {tune_info_sticky['floor_active']})")

In [ ]:
# --- Sample ---
N_SKELETON=10_000
result_boston = sampler_boston.sample(N=N_SKELETON, diagnostics=True)
result_boston_sticky = sampler_boston_sticky.sample(N=N_SKELETON, diagnostics=True)

In [ ]:
from sazz.utils.sampling import resample_pdmp_path, resample_pdmp_path_sticky
from sazz.models.bnn_torch import predict_regression

BURNIN_FRAC = 0.1

# --- Resample ---
samples_np_boston = resample_pdmp_path(
    result_boston["positions"].cpu().numpy(),
    result_boston["velocities"].cpu().numpy(),
    result_boston["times"].cpu().numpy(),
    target_boston.x_ref.cpu().numpy(),
    N_resample=N_RESAMPLE,
    burnin_frac=BURNIN_FRAC,
)
samples_t_boston = torch.tensor(samples_np_boston, dtype=torch.float64)

samples_np_boston_sticky = resample_pdmp_path_sticky(
    result_boston_sticky["positions"].cpu().numpy(),
    result_boston_sticky["velocities"].cpu().numpy(),
    result_boston_sticky["times"].cpu().numpy(),
    target_boston.x_ref.cpu().numpy(),
    N_resample=N_RESAMPLE,
    burnin_frac=BURNIN_FRAC,
)
samples_t_boston_sticky = torch.tensor(samples_np_boston_sticky, dtype=torch.float64)

# --- Predict ---
X_test_boston = datasets["boston"]["X_test"]
y_test_boston = datasets["boston"]["y_test"]

mean_pred_boston, std_pred_boston = predict_regression(
    samples_t_boston, X_test_boston, target_boston
)
mean_pred_boston_sticky, std_pred_boston_sticky = predict_regression(
    samples_t_boston_sticky, X_test_boston, target_boston
)

# --- Metrics ---
noise_std_boston = 0.5 #target_boston.meta["noise_std"]

rmse_boston = ((mean_pred_boston - y_test_boston) ** 2).mean().sqrt()
rmse_boston_sticky = ((mean_pred_boston_sticky - y_test_boston) ** 2).mean().sqrt()

total_std_boston = (std_pred_boston ** 2 + noise_std_boston ** 2).sqrt()
log_lik_boston = (
    -0.5 * ((y_test_boston - mean_pred_boston) / total_std_boston) ** 2
    - total_std_boston.log()
    - 0.5 * torch.log(torch.tensor(2 * torch.pi))
).mean()

total_std_boston_sticky = (std_pred_boston_sticky ** 2 + noise_std_boston ** 2).sqrt()
log_lik_boston_sticky = (
    -0.5 * ((y_test_boston - mean_pred_boston_sticky) / total_std_boston_sticky) ** 2
    - total_std_boston_sticky.log()
    - 0.5 * torch.log(torch.tensor(2 * torch.pi))
).mean()

# --- MAP baseline ---
map_sample_boston = target_boston.x_ref.unsqueeze(0)
mean_map_boston, _ = predict_regression(map_sample_boston, X_test_boston, target_boston)
rmse_map_boston = ((mean_map_boston - y_test_boston) ** 2).mean().sqrt()
residual_std_boston = float(rmse_map_boston)
total_std_map_boston = torch.full_like(mean_map_boston, max(residual_std_boston, noise_std_boston))
log_lik_map_boston = (
    -0.5 * ((y_test_boston - mean_map_boston) / total_std_map_boston) ** 2
    - total_std_map_boston.log()
    - 0.5 * torch.log(torch.tensor(2 * torch.pi))
).mean()

# --- Print ---
print("------------- MAP (Adam) -------------")
print(f"Test RMSE        : {rmse_map_boston:.4f}  (standardised)")
print(f"Test RMSE        : {rmse_map_boston * datasets['boston']['y_std']:.4f}  (original units)")
print(f"Test log-lik     : {log_lik_map_boston:.4f}")
print("------------- Boomerang -------------")
print(f"Test RMSE        : {rmse_boston:.4f}  (standardised)")
print(f"Test RMSE        : {rmse_boston * datasets['boston']['y_std']:.4f}  (original units)")
print(f"Test log-lik     : {log_lik_boston:.4f}")
print(f"Predictive std   : {std_pred_boston.mean():.4f}  (mean across test points)")
print("------------- Sticky Boomerang -------------")
print(f"Test RMSE        : {rmse_boston_sticky:.4f}  (standardised)")
print(f"Test RMSE        : {rmse_boston_sticky * datasets['boston']['y_std']:.4f}  (original units)")
print(f"Test log-lik     : {log_lik_boston_sticky:.4f}")
print(f"Predictive std   : {std_pred_boston_sticky.mean():.4f}  (mean across test points)")

## NAVAL

In [ ]:
# Regression
target_naval = make_bnn_regression(
    datasets['naval']['X_train'], datasets['naval']['y_train'],
    layer_sizes=[datasets['naval']['X_train'].shape[1], 64, 1],
    activation="tanh",
    noise_std=0.1,
)

# Both return a TorchTarget — plug straight into the sampler
sampler_naval = AutomaticBoomerangSampler(
    grad_target=target_naval.grad_target, D=target_naval.D, thinning="pli"
)
sampler_naval.preprocess(x_ref=target_naval.x_ref, Sigma_inv=target_naval.Sigma_inv)

sampler_naval_sticky = StickyAutomaticBoomerangSampler(
    grad_target=target_naval.grad_target, D=target_naval.D, thinning="pli"
)
sampler_naval_sticky.preprocess(x_ref=target_naval.x_ref, Sigma_inv=target_naval.Sigma_inv)

In [ ]:
# --- Sample ---
result_naval = sampler_naval.sample(N=N_SKELETON, diagnostics=True)

result_naval_sticky = sampler_naval_sticky.sample(N=N_SKELETON, diagnostics=True)

In [ ]:
from sazz.utils.sampling import resample_pdmp_path, resample_pdmp_path_sticky
from sazz.models.bnn_torch import predict_regression

BURNIN_FRAC = 0.1

# --- Resample ---
samples_np_naval = resample_pdmp_path(
    result_naval["positions"].cpu().numpy(),
    result_naval["velocities"].cpu().numpy(),
    result_naval["times"].cpu().numpy(),
    target_naval.x_ref.cpu().numpy(),
    N_resample=N_RESAMPLE,
    burnin_frac=BURNIN_FRAC,
)
samples_t_naval = torch.tensor(samples_np_naval, dtype=torch.float64)

samples_np_naval_sticky = resample_pdmp_path_sticky(
    result_naval_sticky["positions"].cpu().numpy(),
    result_naval_sticky["velocities"].cpu().numpy(),
    result_naval_sticky["times"].cpu().numpy(),
    target_naval.x_ref.cpu().numpy(),
    N_resample=N_RESAMPLE,
    burnin_frac=BURNIN_FRAC,
)
samples_t_naval_sticky = torch.tensor(samples_np_naval_sticky, dtype=torch.float64)

# --- Predict ---
X_test_naval = datasets["naval"]["X_test"]
y_test_naval = datasets["naval"]["y_test"]

mean_pred_naval, std_pred_naval = predict_regression(samples_t_naval, X_test_naval, target_naval)
mean_pred_naval_sticky, std_pred_naval_sticky = predict_regression(samples_t_naval_sticky, X_test_naval, target_naval)

# --- Metrics ---
noise_std_naval = 0.1 # target_naval.meta["noise_std"]

rmse_naval = ((mean_pred_naval - y_test_naval) ** 2).mean().sqrt()
rmse_naval_sticky = ((mean_pred_naval_sticky - y_test_naval) ** 2).mean().sqrt()

total_std_naval = (std_pred_naval ** 2 + noise_std_naval ** 2).sqrt()
log_lik_naval = (
    -0.5 * ((y_test_naval - mean_pred_naval) / total_std_naval) ** 2
    - total_std_naval.log()
    - 0.5 * torch.log(torch.tensor(2 * torch.pi))
).mean()

total_std_naval_sticky = (std_pred_naval_sticky ** 2 + noise_std_naval ** 2).sqrt()
log_lik_naval_sticky = (
    -0.5 * ((y_test_naval - mean_pred_naval_sticky) / total_std_naval_sticky) ** 2
    - total_std_naval_sticky.log()
    - 0.5 * torch.log(torch.tensor(2 * torch.pi))
).mean()

# --- MAP baseline ---
map_sample_naval = target_naval.x_ref.unsqueeze(0)
mean_map_naval, _ = predict_regression(map_sample_naval, X_test_naval, target_naval)
rmse_map_naval = ((mean_map_naval - y_test_naval) ** 2).mean().sqrt()
residual_std_naval = float(rmse_map_naval)
total_std_map_naval = torch.full_like(mean_map_naval, max(residual_std_naval, noise_std_naval))
log_lik_map_naval = (
    -0.5 * ((y_test_naval - mean_map_naval) / total_std_map_naval) ** 2
    - total_std_map_naval.log()
    - 0.5 * torch.log(torch.tensor(2 * torch.pi))
).mean()

# --- Print ---
print("------------- MAP (Adam) -------------")
print(f"Test RMSE        : {rmse_map_naval:.4f}  (standardised)")
print(f"Test RMSE        : {rmse_map_naval * datasets['naval']['y_std']:.4f}  (original units)")
print(f"Test log-lik     : {log_lik_map_naval:.4f}")
print("------------- Boomerang -------------")
print(f"Test RMSE        : {rmse_naval:.4f}  (standardised)")
print(f"Test RMSE        : {rmse_naval * datasets['naval']['y_std']:.4f}  (original units)")
print(f"Test log-lik     : {log_lik_naval:.4f}")
print(f"Predictive std   : {std_pred_naval.mean():.4f}  (mean across test points)")
print("------------- Sticky Boomerang -------------")
print(f"Test RMSE        : {rmse_naval_sticky:.4f}  (standardised)")
print(f"Test RMSE        : {rmse_naval_sticky * datasets['naval']['y_std']:.4f}  (original units)")
print(f"Test log-lik     : {log_lik_naval_sticky:.4f}")
print(f"Predictive std   : {std_pred_naval_sticky.mean():.4f}  (mean across test points)")

## ENERGY

In [ ]:
# Regression
target_energy = make_bnn_regression(
    datasets['energy']['X_train'], datasets['energy']['y_train'],
    layer_sizes=[datasets['energy']['X_train'].shape[1], 64, 1],
    activation="tanh",
    noise_std=0.1,
)

# Both return a TorchTarget — plug straight into the sampler
sampler_energy = AutomaticBoomerangSampler(
    grad_target=target_energy.grad_target, D=target_energy.D, thinning="pli"
)
sampler_energy.preprocess(x_ref=target_energy.x_ref, Sigma_inv=target_energy.Sigma_inv)

sampler_energy_sticky = StickyAutomaticBoomerangSampler(
    grad_target=target_energy.grad_target, D=target_energy.D, thinning="pli"
)
sampler_energy_sticky.preprocess(x_ref=target_energy.x_ref, Sigma_inv=target_energy.Sigma_inv)

In [ ]:
# --- Sample ---
result_energy = sampler_energy.sample(N=N_SKELETON, diagnostics=True)

result_energy_sticky = sampler_energy_sticky.sample(N=N_SKELETON, diagnostics=True)

In [ ]:
from sazz.utils.sampling import resample_pdmp_path, resample_pdmp_path_sticky
from sazz.models.bnn_torch import predict_regression

BURNIN_FRAC = 0.1

# --- Resample ---
samples_np_energy = resample_pdmp_path(
    result_energy["positions"].cpu().numpy(),
    result_energy["velocities"].cpu().numpy(),
    result_energy["times"].cpu().numpy(),
    target_energy.x_ref.cpu().numpy(),
    N_resample=N_RESAMPLE,
    burnin_frac=BURNIN_FRAC,
)
samples_t_energy = torch.tensor(samples_np_energy, dtype=torch.float64)

samples_np_energy_sticky = resample_pdmp_path_sticky(
    result_energy_sticky["positions"].cpu().numpy(),
    result_energy_sticky["velocities"].cpu().numpy(),
    result_energy_sticky["times"].cpu().numpy(),
    target_energy.x_ref.cpu().numpy(),
    N_resample=N_RESAMPLE,
    burnin_frac=BURNIN_FRAC,
)
samples_t_energy_sticky = torch.tensor(samples_np_energy_sticky, dtype=torch.float64)

# --- Predict ---
X_test_energy = datasets["energy"]["X_test"]
y_test_energy = datasets["energy"]["y_test"]

mean_pred_energy, std_pred_energy = predict_regression(samples_t_energy, X_test_energy, target_energy)
mean_pred_energy_sticky, std_pred_energy_sticky = predict_regression(samples_t_energy_sticky, X_test_energy, target_energy)

# --- Metrics ---
noise_std_energy = 0.1 #target_energy.meta["noise_std"]

rmse_energy = ((mean_pred_energy - y_test_energy) ** 2).mean().sqrt()
rmse_energy_sticky = ((mean_pred_energy_sticky - y_test_energy) ** 2).mean().sqrt()

total_std_energy = (std_pred_energy ** 2 + noise_std_energy ** 2).sqrt()
log_lik_energy = (
    -0.5 * ((y_test_energy - mean_pred_energy) / total_std_energy) ** 2
    - total_std_energy.log()
    - 0.5 * torch.log(torch.tensor(2 * torch.pi))
).mean()

total_std_energy_sticky = (std_pred_energy_sticky ** 2 + noise_std_energy ** 2).sqrt()
log_lik_energy_sticky = (
    -0.5 * ((y_test_energy - mean_pred_energy_sticky) / total_std_energy_sticky) ** 2
    - total_std_energy_sticky.log()
    - 0.5 * torch.log(torch.tensor(2 * torch.pi))
).mean()

# --- MAP baseline ---
map_sample_energy = target_energy.x_ref.unsqueeze(0)
mean_map_energy, _ = predict_regression(map_sample_energy, X_test_energy, target_energy)
rmse_map_energy = ((mean_map_energy - y_test_energy) ** 2).mean().sqrt()
residual_std_energy = float(rmse_map_energy)
total_std_map_energy = torch.full_like(mean_map_energy, max(residual_std_energy, noise_std_energy))
log_lik_map_energy = (
    -0.5 * ((y_test_energy - mean_map_energy) / total_std_map_energy) ** 2
    - total_std_map_energy.log()
    - 0.5 * torch.log(torch.tensor(2 * torch.pi))
).mean()

# --- Print ---
print("------------- MAP (Adam) -------------")
print(f"Test RMSE        : {rmse_map_energy:.4f}  (standardised)")
print(f"Test RMSE        : {rmse_map_energy * datasets['energy']['y_std']:.4f}  (original units)")
print(f"Test log-lik     : {log_lik_map_energy:.4f}")
print("------------- Boomerang -------------")
print(f"Test RMSE        : {rmse_energy:.4f}  (standardised)")
print(f"Test RMSE        : {rmse_energy * datasets['energy']['y_std']:.4f}  (original units)")
print(f"Test log-lik     : {log_lik_energy:.4f}")
print(f"Predictive std   : {std_pred_energy.mean():.4f}  (mean across test points)")
print("------------- Sticky Boomerang -------------")
print(f"Test RMSE        : {rmse_energy_sticky:.4f}  (standardised)")
print(f"Test RMSE        : {rmse_energy_sticky * datasets['energy']['y_std']:.4f}  (original units)")
print(f"Test log-lik     : {log_lik_energy_sticky:.4f}")
print(f"Predictive std   : {std_pred_energy_sticky.mean():.4f}  (mean across test points)")

## PIMA INDIAN DATA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("benchmarks_august/datasets/diabetes.csv")
X = df.drop("Outcome", axis=1).values.astype(float)
y = df["Outcome"].values.astype(float)

# Handle implicit missingness: zero means missing in these columns
for col_idx in [1, 2, 3, 4, 5]:  # Glucose, BP, Skin, Insulin, BMI
    mask = X[:, col_idx] == 0
    X[mask, col_idx] = np.nan
X = np.where(np.isnan(X), np.nanmean(X, axis=0), X)  # mean impute

# Standardize
X = (X - X.mean(axis=0)) / X.std(axis=0)

X_train_pima, X_test_pima, y_train_pima, y_test_pima = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize using training statistics
x_mean_pima, x_std_pima = X_train_pima.mean(axis=0), X_train_pima.std(axis=0)
x_std_pima[x_std_pima == 0] = 1.0   # guard against constant columns (Naval has some)
y_mean_pima, y_std_pima = y_train_pima.mean(), y_train_pima.std()

In [ ]:
datasets['pima'] = {
        "X_train": torch.tensor(X_train_pima, dtype=torch.float64),
        "y_train": torch.tensor(y_train_pima, dtype=torch.float64),
        "X_test":  torch.tensor(X_test_pima, dtype=torch.float64),
        "y_test":  torch.tensor(y_test_pima, dtype=torch.float64),
        # keep numpy versions for sklearn metrics if needed
        "x_mean": x_mean_pima, "x_std": x_std_pima,
        "y_mean": y_mean_pima, "y_std": y_std_pima,
}

In [ ]:
from sazz.models.bnn_torch import make_bnn_classification

target_pima = make_bnn_classification(
    datasets['pima']['X_train'], datasets['pima']['y_train'],
    layer_sizes=[datasets['pima']['X_train'].shape[1], 64, 1],
    activation="relu",
)

In [ ]:
sampler_pima = AutomaticBoomerangSampler(
    grad_target=target_pima.grad_target, D=target_pima.D, thinning="pli"
)
sampler_pima.preprocess(x_ref=target_pima.x_ref, Sigma_inv=target_pima.Sigma_inv)

sampler_pima_sticky = StickyAutomaticBoomerangSampler(
    grad_target=target_pima.grad_target, D=target_pima.D, thinning="pli"
)
sampler_pima_sticky.preprocess(x_ref=target_pima.x_ref, Sigma_inv=target_pima.Sigma_inv)

In [ ]:
# --- Sample ---
result_pima = sampler_pima.sample(N=N_SKELETON, diagnostics=True)

result_pima_sticky = sampler_pima_sticky.sample(N=N_SKELETON, diagnostics=True)

In [ ]:
from sazz.utils.sampling import resample_pdmp_path, resample_pdmp_path_sticky
from sazz.models.bnn_torch import predict_classification

BURNIN_FRAC = 0.1

# --- Resample ---
samples_np_pima = resample_pdmp_path(
    result_pima["positions"].cpu().numpy(),
    result_pima["velocities"].cpu().numpy(),
    result_pima["times"].cpu().numpy(),
    target_pima.x_ref.cpu().numpy(),
    N_resample=N_RESAMPLE,
    burnin_frac=BURNIN_FRAC,
)
samples_t_pima = torch.tensor(samples_np_pima, dtype=torch.float64)

samples_np_pima_sticky = resample_pdmp_path_sticky(
    result_pima_sticky["positions"].cpu().numpy(),
    result_pima_sticky["velocities"].cpu().numpy(),
    result_pima_sticky["times"].cpu().numpy(),
    target_pima.x_ref.cpu().numpy(),
    N_resample=N_RESAMPLE,
    burnin_frac=BURNIN_FRAC,
)
samples_t_pima_sticky = torch.tensor(samples_np_pima_sticky, dtype=torch.float64)

# --- Predict ---
X_test_pima = datasets["pima"]["X_test"]
y_test_pima = datasets["pima"]["y_test"]

probs_pima, entropy_pima = predict_classification(samples_t_pima, X_test_pima, target_pima)
preds_pima = (probs_pima[:, 1] > 0.5).long()
accuracy_pima = (preds_pima == y_test_pima.long()).float().mean()
p_correct_pima = probs_pima[torch.arange(len(y_test_pima)), y_test_pima.long()]
log_lik_pima = p_correct_pima.clamp(min=1e-12).log().mean()

probs_pima_sticky, entropy_pima_sticky = predict_classification(samples_t_pima_sticky, X_test_pima, target_pima)
preds_pima_sticky = (probs_pima_sticky[:, 1] > 0.5).long()
accuracy_pima_sticky = (preds_pima_sticky == y_test_pima.long()).float().mean()
p_correct_pima_sticky = probs_pima_sticky[torch.arange(len(y_test_pima)), y_test_pima.long()]
log_lik_pima_sticky = p_correct_pima_sticky.clamp(min=1e-12).log().mean()

map_sample_pima = target_pima.x_ref.unsqueeze(0)
probs_map_pima, _ = predict_classification(map_sample_pima, X_test_pima, target_pima)
preds_map_pima = (probs_map_pima[:, 1] > 0.5).long()
accuracy_map_pima = (preds_map_pima == y_test_pima.long()).float().mean()
p_correct_map_pima = probs_map_pima[torch.arange(len(y_test_pima)), y_test_pima.long()]
log_lik_map_pima = p_correct_map_pima.clamp(min=1e-12).log().mean()

print("------------- MAP (Adam) -------------")
print(f"Accuracy         : {accuracy_map_pima:.4f}")
print(f"Test log-lik     : {log_lik_map_pima:.4f}")
print("------------- Boomerang -------------")
print(f"Accuracy         : {accuracy_pima:.4f}")
print(f"Test log-lik     : {log_lik_pima:.4f}")
print(f"Mean entropy     : {entropy_pima.mean():.4f}")
print("------------- Sticky Boomerang -------------")
print(f"Accuracy         : {accuracy_pima_sticky:.4f}")
print(f"Test log-lik     : {log_lik_pima_sticky:.4f}")
print(f"Mean entropy     : {entropy_pima_sticky.mean():.4f}")